<a href="https://colab.research.google.com/github/abhimanyu1502/flyrank1st-assignment/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Ranked Actions + Reason Codes

We translate our K-Means ($K=5$) clustering output into a prioritized content review queue. Each page receives:
1. **Assigned Archetype Persona**
2. **Recommended Action Playbook**
3. **Transparent Reason Code** (`STALE_HIGH_REACH`, `PAGE2_HIGH_CTR`, `TOP_PERFORMING_CHAMPION`, `HIGH_BOUNCE_LOW_ENGAGEMENT`, `LOW_ORGANIC_DEMAND`)
4. **Action Priority Score** (combining reach, ranking inverted score, and staleness urgency).

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from IPython.display import display, HTML

# 1. Dynamic path resolution across environments
possible_paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "/content/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
]
data_path = next((p for p in possible_paths if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

df_raw = pd.read_csv(data_path)
df_clean = df_raw[(df_raw['impressions_90d'] >= 10) & (df_raw['content_age_days'] >= 90)].copy()

# 2. Train Clustering Model
features = ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'engagement_rate']
X = df_clean[features].copy()
X['impressions_log'] = np.log1p(X['impressions_90d'])
X['staleness_log'] = np.log1p(X['days_since_last_update'])
scaled_features = ['impressions_log', 'avg_position', 'ctr', 'staleness_log', 'engagement_rate']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X[scaled_features])
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_clean['cluster'] = kmeans.fit_predict(X_scaled)

# 3. Archetype Mapping
archetype_names = {
    0: 'Champions',
    1: 'Stale High-Reach',
    2: 'Hidden Gems',
    3: 'Low Demand',
    4: 'Low Engagement'
}
df_clean['archetype'] = df_clean['cluster'].map(archetype_names)

# 4. Action Mapping
action_map = {
    'Champions': 'Monitor & Protect Rankings',
    'Stale High-Reach': 'Schedule Fact Check & Content Refresh',
    'Hidden Gems': 'Optimize Title & Meta Description',
    'Low Demand': 'Consolidate or Prune',
    'Low Engagement': 'Review Content Intent & Readability'
}
df_clean['recommended_action'] = df_clean['archetype'].map(action_map)

# 5. Reason Code Generator
def get_reason_code(row):
    if row['archetype'] == 'Stale High-Reach':
        return 'STALE_HIGH_REACH'
    elif row['archetype'] == 'Hidden Gems':
        return 'PAGE2_HIGH_CTR'
    elif row['archetype'] == 'Champions':
        return 'TOP_PERFORMING_CHAMPION'
    elif row['archetype'] == 'Low Engagement':
        return 'HIGH_BOUNCE_LOW_ENGAGEMENT'
    else:
        return 'LOW_ORGANIC_DEMAND'

df_clean['reason_code'] = df_clean.apply(get_reason_code, axis=1)

# 6. Composite Action Priority Score
df_clean['action_priority_score'] = (
    0.40 * (np.log1p(df_clean['impressions_90d']) / np.log1p(df_clean['impressions_90d'].max())) +
    0.35 * (1.0 / (df_clean['avg_position'] + 1)) +
    0.25 * (np.clip(df_clean['days_since_last_update'] / 365.0, 0, 1))
).round(4)

df_ranked = df_clean.sort_values('action_priority_score', ascending=False).reset_index(drop=True)
df_ranked['rank'] = df_ranked.index + 1

preview_cols = ['rank', 'archetype', 'recommended_action', 'reason_code', 'action_priority_score', 'impressions_90d', 'avg_position', 'days_since_last_update']
print("=== TOP 5 RANKED ACTIONS IN PLAYBOOK ===")
display(HTML(df_ranked[preview_cols].head(5).to_html(index=False)))

=== TOP 5 RANKED ACTIONS IN PLAYBOOK ===


rank,archetype,recommended_action,reason_code,action_priority_score,impressions_90d,avg_position,days_since_last_update
1,Stale High-Reach,Schedule Fact Check & Content Refresh,STALE_HIGH_REACH,0.6031,2695,0.2,104
2,Stale High-Reach,Schedule Fact Check & Content Refresh,STALE_HIGH_REACH,0.6019,43650,0.7,104
3,Stale High-Reach,Schedule Fact Check & Content Refresh,STALE_HIGH_REACH,0.5722,309192,2.0,104
4,Stale High-Reach,Schedule Fact Check & Content Refresh,STALE_HIGH_REACH,0.5529,143314,1.9,104
5,Champions,Monitor & Protect Rankings,TOP_PERFORMING_CHAMPION,0.5442,312694,1.4,20


### 2. Intended Use and Limits

#### Intended Use:
- **Target Persona**: Content Strategists, Managing Editors, and Technical SEO Leads.
- **Workflow Integration**: Triage tool used during sprint planning to pull the top 20–50 priority items per client each week into Jira/Asana workflows.
- **Granularity**: URL-level diagnostic recommendations based on multi-dimensional performance patterns.

#### Operational Limits:
- **Immature Content**: Pages younger than 90 days are excluded because early ranking volatility introduces false staleness signals.
- **Zero-Impression Pages**: Pages with $<10$ impressions across 90 days have insufficient telemetry for meaningful clustering.
- **Non-Search Pages**: Functional pages (e.g. `/cart`, `/privacy-policy`, `/login`) must be filtered out at the CMS level.

In [ ]:
# Intended Use & Coverage Boundary Summary
coverage_summary = pd.DataFrame({
    'Portfolio Segment': ['In-Scope Contract Portfolio', 'Excluded Young Pages (<90d)', 'Excluded Low-Demand Pages (<10 imp)'],
    'Row Count': [len(df_clean), (df_raw['content_age_days'] < 90).sum(), (df_raw['impressions_90d'] < 10).sum()],
    'Playbook Status': ['[ACTIVE PLAYBOOK ROUTING]', '[BYPASS: AWAIT MATURITY]', '[BYPASS: PRUNE OR CONSOLIDATE]']
})

print("=== OPERATIONAL SCOPE & COVERAGE AUDIT ===")
print(coverage_summary.to_string(index=False))

#### Mandatory Human Review Checklist Before Action:
1. **SERP Query Intent**: Confirm whether top search queries for this URL remain relevant to the current page topic.
2. **Branded vs. Non-Branded Queries**: Check if search impressions originate from branded queries outside editorial control.
3. **URL Redirection History**: Confirm the URL has not been recently redirected or migrated.

#### 🛑 The Strict NO-GO List (What Should NEVER Be Automated):
- 🚫 **NO Auto-Deletions**: Never automatically delete or unpublish low-demand pages without manual backlink and revenue checks.
- 🚫 **NO Autonomous YMYL Updates**: Never auto-publish AI-generated text on medical, financial, or legal advice pages.
- 🚫 **NO Uninspected Bulk 301 Redirects**: Never automate redirects based solely on low impression metrics.

In [4]:
# Programmatic Human Governance Protocol Table
governance_rules = pd.DataFrame({
    'Automated ML Recommendation': ['Schedule Fact Check & Content Refresh', 'Optimize Title & Meta Description', 'Consolidate or Prune', 'Review Content Intent & Readability'],
    'Required Human Review Step': ['Editor verifies factual accuracy & updates outdated stats', 'SEO specialist audits title tag against primary keyword intent', 'Strategist reviews backlink profile and internal link graph', 'Content writer assesses introduction hook and layout readability'],
    'Automation Level': ['Decision-Support Only', 'Decision-Support Only', 'Strict Human Gatekeeper', 'Decision-Support Only']
})

print("=== HUMAN GOVERNANCE & NO-GO AUDIT PROTOCOL ===")
from IPython.display import display, HTML
display(HTML(governance_rules.to_html(index=False)))

=== HUMAN GOVERNANCE & NO-GO AUDIT PROTOCOL ===


Automated ML Recommendation,Required Human Review Step,Automation Level
Schedule Fact Check & Content Refresh,Editor verifies factual accuracy & updates outdated stats,Decision-Support Only
Optimize Title & Meta Description,SEO specialist audits title tag against primary keyword intent,Decision-Support Only
Consolidate or Prune,Strategist reviews backlink profile and internal link graph,Strict Human Gatekeeper
Review Content Intent & Readability,Content writer assesses introduction hook and layout readability,Decision-Support Only


### 4. Monitoring / Retrain Triggers

We establish four clear operational triggers that indicate model outputs have gone stale and require retraining:

1. **Monthly Ingestion Cadence**: Re-run clustering pipeline on the 1st of every calendar month with rolling 90-day data.
2. **Silhouette Degradation Trigger**: If out-of-sample Silhouette score drops below **0.25**, retrain and evaluate optimal $K$.
3. **SERP Algorithm Update Trigger**: Trigger immediate recalibration following major confirmed Google Core Updates.
4. **Centroid Drift Trigger**: If any cluster centroid coordinates shift by $>0.5$ standard deviations, update archetype boundary thresholds.

In [6]:
# Monitoring & Retraining Threshold Configuration
retrain_triggers = pd.DataFrame({
    'Monitoring Metric': ['Cadence Trigger', 'Model Stability', 'Data Distribution Drift', 'External Shock'],
    'Condition for Retraining': ['Monthly rolling window update (1st of month)', 'Silhouette Score < 0.25 on new monthly extract', 'Centroid shift > 0.5 sigma in normalized space', 'Confirmed Google Search Core Algorithm Update'],
    'Action Required': ['Re-run pipeline & refresh queue', 'Re-tune K & re-cluster', 'Update centroid coordinates & playbook tags', 'Full model re-validation']
})

print("=== MODEL MONITORING & RETRAINING TRIGGERS ===")
from IPython.display import display, HTML
display(HTML(retrain_triggers.to_html(index=False)))

=== MODEL MONITORING & RETRAINING TRIGGERS ===


Monitoring Metric,Condition for Retraining,Action Required
Cadence Trigger,Monthly rolling window update (1st of month),Re-run pipeline & refresh queue
Model Stability,Silhouette Score < 0.25 on new monthly extract,Re-tune K & re-cluster
Data Distribution Drift,Centroid shift > 0.5 sigma in normalized space,Update centroid coordinates & playbook tags
External Shock,Confirmed Google Search Core Algorithm Update,Full model re-validation


### 5. Exports for the Paper

We export the finalized deliverables to `work/outputs/`:
1. `work/outputs/content_action_playbook_queue.csv` — Full prioritized action queue (26,254 rows).
2. `work/outputs/playbook_archetype_summary.csv` — Aggregated archetype distributions and operational metrics for the Capstone paper.

In [8]:
# 1. Prepare export directories
output_dir = "../../work/outputs"
os.makedirs(output_dir, exist_ok=True)

# 2. Export full ranked queue
export_cols = [
    'rank', 'content_id', 'client_id', 'archetype',
    'recommended_action', 'reason_code', 'action_priority_score',
    'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'engagement_rate'
]

queue_path = f"{output_dir}/content_action_playbook_queue.csv"
df_ranked[export_cols].to_csv(queue_path, index=False)

# 3. Export aggregated archetype summary
summary_table = df_ranked.groupby('archetype').agg(
    total_pages=('content_id', 'count'),
    pct_of_corpus=('content_id', lambda x: (len(x) / len(df_ranked)) * 100),
    median_impressions=('impressions_90d', 'median'),
    median_position=('avg_position', 'median'),
    median_staleness=('days_since_last_update', 'median'),
    primary_action=('recommended_action', 'first')
).round(1)

summary_path = f"{output_dir}/playbook_archetype_summary.csv"
summary_table.to_csv(summary_path)

print("=== ARTIFACT EXPORT AUDIT ===")
print(f"[EXPORTED] Full Ranked Queue:     {queue_path} ({len(df_ranked):,} rows)")
print(f"[EXPORTED] Archetype Summary:     {summary_path} ({len(summary_table)} archetypes)")
print("\nTop 5 preview of exported queue:")
from IPython.display import display, HTML
display(HTML(df_ranked[export_cols].head(5).to_html(index=False)))

=== ARTIFACT EXPORT AUDIT ===
[EXPORTED] Full Ranked Queue:     ../../work/outputs/content_action_playbook_queue.csv (26,254 rows)
[EXPORTED] Archetype Summary:     ../../work/outputs/playbook_archetype_summary.csv (5 archetypes)

Top 5 preview of exported queue:


rank,content_id,client_id,archetype,recommended_action,reason_code,action_priority_score,impressions_90d,avg_position,ctr,days_since_last_update,engagement_rate
1,content_7247c9f3c142,client_19581e27de,Stale High-Reach,Schedule Fact Check & Content Refresh,STALE_HIGH_REACH,0.6031,2695,0.2,0.07,104,0.00
2,content_7a6df559322d,client_19581e27de,Stale High-Reach,Schedule Fact Check & Content Refresh,STALE_HIGH_REACH,0.6019,43650,0.7,0.14,104,11.59
3,content_9532f197bbc8,client_4e07408562,Stale High-Reach,Schedule Fact Check & Content Refresh,STALE_HIGH_REACH,0.5722,309192,2.0,0.87,104,8.01
4,content_03d2673b2553,client_19581e27de,Stale High-Reach,Schedule Fact Check & Content Refresh,STALE_HIGH_REACH,0.5529,143314,1.9,0.83,104,0.94
5,content_44e481c8f55b,client_19581e27de,Champions,Monitor & Protect Rankings,TOP_PERFORMING_CHAMPION,0.5442,312694,1.4,0.65,20,0.76


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.